In [9]:
# Import all PySpark functions
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.option("header","true")\
               .option("inferSchema", "true")\
               .csv("Files/Rawdata/DataCoSupplyChainDataset.csv")

print(f"Total_Column: {len(df.columns)}")
print(f"Total_row: {df.count()}")
df.printSchema()

# Step 1: Clean column names 
def clean_column_name(name):
    return name.replace(" ", "_") \
               .replace("(", "") \
               .replace(")", "") \
               .replace("/", "_")

#Step 2: Apply clean_name
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, clean_column_name(col_name))

#Step 3 Fix date columns
df = df.withColumn("Order_Date", to_timestamp("order_date_DateOrders", "M/d/yyyy H:mm")) \
       .withColumn("Shipping_Date", to_timestamp("shipping_date_DateOrders", "M/d/yyyy H:mm"))

#Step 4 drop unnecessary columns
df = df.drop("Customer_Password", "Product_Description", "Product_Image", 
             "order_date_DateOrders", "shipping_date_DateOrders")

# Step 5: Check result
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")
df.printSchema()

# Save cleaned data as Delta table
df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("silver_supply_chain")

print("Silver table created successfully!")

#: Sales by Market
gold_sales_market = df.groupBy("Market") \
    .agg(
        count("Order_Id").alias("Total_Orders"),
        round(sum("Sales"), 2).alias("Total_Sales"),
        round(sum("Order_Profit_Per_Order"), 2).alias("Total_Profit"),
        round(avg("Order_Profit_Per_Order"), 2).alias("Avg_Profit_Per_Order")
    ).orderBy("Total_Sales", ascending=False)

gold_sales_market.show()

#  Table 1
gold_sales_market.write.format("delta") \
                 .mode("overwrite") \
                 .saveAsTable("gold_sales_by_market")

# : Late Delivery by Shipping Mode
gold_delivery = df.groupBy("Shipping_Mode") \
    .agg(
        count("Order_Id").alias("Total_Orders"),
        count(when(col("Delivery_Status") == "Late delivery", 1)).alias("Late_Orders"),
        round(avg("Days_for_shipping_real"), 2).alias("Avg_Actual_Days"),
        round(avg("Days_for_shipment_scheduled"), 2).alias("Avg_Scheduled_Days")
    ).orderBy("Late_Orders", ascending=False)

gold_delivery.show()

# Table 2
gold_delivery.write.format("delta") \
             .mode("overwrite") \
             .saveAsTable("gold_delivery_by_shipping")

print("Gold tables saved successfully!")

StatementMeta(, 53de4b31-96ec-4c7b-9c43-51a2ead76760, 53, Finished, Available, Finished, False)

Total_Column: 53
Total_row: 180519
root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = true)
 |-- 